In [10]:
"""
Black-Scholes Future Volatility Calculation - Nifty Options
--------------------------------------------------------------
Computes implied (future) volatility from market option prices, for
both CALL and PUT sides, at every strike, for the 1M / 3M / 6M NSE
option chain files. Also plots volatility vs strike for each.

Model used: Black-Scholes Option Pricing Model
Parameters computed per row: S (spot), K (strike), T (time to expiry),
r (risk-free rate), sigma (implied volatility - solved numerically).

Install first:
    pip install pandas numpy scipy matplotlib
"""

import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
import matplotlib.pyplot as plt

RISK_FREE_RATE = 0.065  # r - assumed India risk-free rate

FILES = {
    "1M": {"path": "/content/NSE 1st Month.csv", "T": 1 / 12},
    "3M": {"path": "/content/NSE 3rd Month.csv", "T": 3 / 12},
    "6M": {"path": "/content/NSE 6th  Month.csv", "T": 6 / 12},
}


# ---------- Step 1 & 2: Read and clean the CSV ----------

def load_and_clean(filepath):
    df = pd.read_csv(filepath, skiprows=2, header=None)

    columns = [
        "BLANK1",
        "CE_OI", "CE_CHNG_IN_OI", "CE_VOLUME", "CE_IV", "CE_LTP", "CE_CHNG",
        "CE_BID_QTY", "CE_BID", "CE_ASK", "CE_ASK_QTY",
        "STRIKE",
        "PE_BID_QTY", "PE_BID", "PE_ASK", "PE_ASK_QTY",
        "PE_CHNG", "PE_LTP", "PE_IV", "PE_VOLUME", "PE_CHNG_IN_OI", "PE_OI",
        "BLANK2",
    ]
    df.columns = columns[:len(df.columns)]
    df = df.drop(columns=[c for c in ["BLANK1", "BLANK2"] if c in df.columns])

    # Clean numeric columns: remove commas, treat "-" as missing
    for col in df.columns:
        df[col] = (
            df[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df[col] = df[col].replace("-", np.nan)
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["STRIKE"]).reset_index(drop=True)
    return df


# ---------- Parameter: S (spot) via Put-Call Parity ----------

def estimate_spot(df):
    valid = df.dropna(subset=["CE_LTP", "PE_LTP"]).copy()
    valid["diff"] = (valid["CE_LTP"] - valid["PE_LTP"]).abs()
    atm_row = valid.loc[valid["diff"].idxmin()]
    S = atm_row["CE_LTP"] - atm_row["PE_LTP"] + atm_row["STRIKE"]
    return S


# ---------- Black-Scholes pricing formulas ----------

def bs_call_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + sigma ** 2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + sigma ** 2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)


# ---------- Solve for sigma (implied / future volatility) ----------

def implied_vol(price, S, K, T, r, option_type="call"):
    if price is None or np.isnan(price) or price <= 0:
        return np.nan

    price_func = bs_call_price if option_type == "call" else bs_put_price

    def objective(sigma):
        return price_func(S, K, T, r, sigma) - price

    try:
        return brentq(objective, 1e-6, 5.0, maxiter=500)
    except ValueError:
        return np.nan


# ---------- Main processing per file ----------

def process_file(label, path, T):
    df = load_and_clean(path)
    S = estimate_spot(df)

    df["Calculated_CE_IV"] = df.apply(
        lambda row: implied_vol(row["CE_LTP"], S, row["STRIKE"], T,
                                 RISK_FREE_RATE, "call"),
        axis=1,
    )
    df["Calculated_PE_IV"] = df.apply(
        lambda row: implied_vol(row["PE_LTP"], S, row["STRIKE"], T,
                                 RISK_FREE_RATE, "put"),
        axis=1,
    )

    # Convert to percentage for readability
    df["Calculated_CE_IV_%"] = df["Calculated_CE_IV"] * 100
    df["Calculated_PE_IV_%"] = df["Calculated_PE_IV"] * 100

    output_csv = f"/mnt/user-data/outputs/nifty_{label.lower()}_black_scholes.csv"
    df.to_csv(output_csv, index=False)

    print(f"{label}: S (estimated spot) = {S:.2f}, T = {T:.4f} years, "
          f"r = {RISK_FREE_RATE}, rows processed = {len(df)}")
    print(f"  Saved: {output_csv}")

    return df, S


# ---------- Visualization ----------

def plot_volatility(results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for ax, (label, (df, S)) in zip(axes, results.items()):
        ax.plot(df["STRIKE"], df["Calculated_CE_IV_%"], "o-",
                label="Call IV", color="tab:blue", markersize=3)
        ax.plot(df["STRIKE"], df["Calculated_PE_IV_%"], "o-",
                label="Put IV", color="tab:red", markersize=3)
        ax.axvline(S, color="gray", linestyle="--", linewidth=1, label="Spot (S)")
        ax.set_title(f"{label} Expiry - Future Volatility vs Strike")
        ax.set_xlabel("Strike Price")
        ax.legend()
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Implied (Future) Volatility %")
    plt.tight_layout()

    output_img = "/mnt/user-data/outputs/nifty_future_volatility_smile.png"
    plt.savefig(output_img, dpi=150)
    print(f"Saved chart: {output_img}")
    plt.close()


def main():
    results = {}
    for label, info in FILES.items():
        df, S = process_file(label, info["path"], info["T"])
        results[label] = (df, S)

    plot_volatility(results)


if __name__ == "__main__":
    main()


1M: S (estimated spot) = 23261.60, T = 0.0833 years, r = 0.065, rows processed = 129
  Saved: /mnt/user-data/outputs/nifty_1m_black_scholes.csv
3M: S (estimated spot) = 23544.30, T = 0.2500 years, r = 0.065, rows processed = 28
  Saved: /mnt/user-data/outputs/nifty_3m_black_scholes.csv
6M: S (estimated spot) = 23893.10, T = 0.5000 years, r = 0.065, rows processed = 13
  Saved: /mnt/user-data/outputs/nifty_6m_black_scholes.csv
Saved chart: /mnt/user-data/outputs/nifty_future_volatility_smile.png
